# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [6]:
!pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa soundfile

# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [7]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})

In [5]:
!pip uninstall -y torchcodec
!pip install -qU datasets librosa soundfile ffmpeg

Found existing installation: torchcodec 0.14.0
Uninstalling torchcodec-0.14.0:
  Successfully uninstalled torchcodec-0.14.0


In [12]:
from datasets import load_dataset
from datasets.features import Audio
import io
import soundfile as sf
import librosa
import numpy as np

data = load_dataset("ashraq/esc50", split="train")
data = data.cast_column("audio", Audio(decode=False))

audios = []
labels = []

for item in data:
    audio_bytes = item["audio"]["bytes"]
    
    y, sr = sf.read(io.BytesIO(audio_bytes))
    
    if y.ndim > 1:
        y = y.mean(axis=1)
    
    y = y.astype("float32")
    
    if sr != 32000:
        y = librosa.resample(y, orig_sr=sr, target_sr=32000)
        sr = 32000
    
    audios.append(y)
    labels.append(item["category"])

print(len(audios))
print(labels[:10])
print(audios[0].shape)

Repo card metadata block was not found. Setting CardData to empty.


2000
['dog', 'chirping_birds', 'vacuum_cleaner', 'vacuum_cleaner', 'thunderstorm', 'thunderstorm', 'door_wood_knock', 'can_opening', 'crow', 'door_wood_knock']
(160000,)


In [14]:
import os
import urllib.request

panns_dir = "/Users/hduarte/panns_data"
os.makedirs(panns_dir, exist_ok=True)

url = "https://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv"
file_path = os.path.join(panns_dir, "class_labels_indices.csv")

urllib.request.urlretrieve(url, file_path)

print("Downloaded:", file_path)

Downloaded: /Users/hduarte/panns_data/class_labels_indices.csv


In [17]:
import os
import urllib.request

os.makedirs("/Users/hduarte/panns_data", exist_ok=True)

urllib.request.urlretrieve(
    "https://zenodo.org/records/3987831/files/Cnn14_mAP%3D0.431.pth?download=1",
    "/Users/hduarte/panns_data/Cnn14_mAP=0.431.pth"
)

('/Users/hduarte/panns_data/Cnn14_mAP=0.431.pth',
 <http.client.HTTPMessage at 0x377dbd150>)

In [18]:
os.path.exists("/Users/hduarte/panns_data/Cnn14_mAP=0.431.pth")

True

In [19]:
from panns_inference import AudioTagging

at = AudioTagging(
    checkpoint_path="/Users/hduarte/panns_data/Cnn14_mAP=0.431.pth",
    device="cpu"
)

Checkpoint path: /Users/hduarte/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


In [20]:
import torch
import panns_inference

print(torch.__version__)
print(panns_inference.__version__)

2.12.0
0.1.0


The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [21]:
import io
import soundfile as sf

audio_bytes = data[0]["audio"]["bytes"]

y, sr = sf.read(io.BytesIO(audio_bytes))

print(y.shape)
print(sr)

(220500,)
44100


In [22]:
import numpy as np

clip = y.astype(np.float32)

embedding, prediction = at.inference(
    clip.reshape(1, -1)
)

print(embedding.shape)

(1, 527)


In [23]:
prediction, embedding = at.inference(
    clip.reshape(1, -1)
)

print("prediction:", prediction.shape)
print("embedding:", embedding.shape)

prediction: (1, 527)
embedding: (1, 2048)


In [24]:
embeddings = []

for i in range(100):
    audio_bytes = data[i]["audio"]["bytes"]
    
    y, sr = sf.read(io.BytesIO(audio_bytes))
    
    if y.ndim > 1:
        y = y.mean(axis=1)

    y = y.astype(np.float32)

    prediction, embedding = at.inference(y.reshape(1, -1))

    embeddings.append(embedding[0])

print(len(embeddings))
print(embeddings[0].shape)

100
(2048,)


In [25]:
embeddings = [e.tolist() for e in embeddings]

print(type(embeddings[0]))
print(len(embeddings[0]))

<class 'list'>
2048


In [26]:
items = []

for i in range(len(embeddings)):
    items.append({
        "id": str(i),
        "values": embeddings[i],
        "metadata": {
            "label": labels[i]
        }
    })

items[0].keys()

dict_keys(['id', 'values', 'metadata'])

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=)

In [29]:
index_name = "music-recommendations"

existing_indexes = [index["name"] for index in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=2048,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [30]:
index.upsert(vectors=items)

{'upserted_count': 100}

In [31]:
query_vector = embeddings[0]

results = index.query(
    vector=query_vector,
    top_k=5,
    include_metadata=True
)

results

{'matches': [{'id': '0',
              'metadata': {'label': 'dog'},
              'score': 1.00012076,
              'values': []},
             {'id': '77',
              'metadata': {'label': 'mouse_click'},
              'score': 0.876694322,
              'values': []},
             {'id': '14',
              'metadata': {'label': 'dog'},
              'score': 0.777210832,
              'values': []},
             {'id': '85',
              'metadata': {'label': 'coughing'},
              'score': 0.767594755,
              'values': []},
             {'id': '84',
              'metadata': {'label': 'coughing'},
              'score': 0.754443586,
              'values': []}],
 'namespace': '',
 'usage': {'read_units': 1}}

In [32]:
for match in results["matches"]:
    print("ID:", match["id"])
    print("Score:", match["score"])
    print("Label:", match["metadata"]["label"])
    print("---")

ID: 0
Score: 1.00012076
Label: dog
---
ID: 77
Score: 0.876694322
Label: mouse_click
---
ID: 14
Score: 0.777210832
Label: dog
---
ID: 85
Score: 0.767594755
Label: coughing
---
ID: 84
Score: 0.754443586
Label: coughing
---


In [33]:
def recommend_similar_audio(audio_id, top_k=5):
    query_vector = embeddings[audio_id]

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    print(f"Query audio ID: {audio_id}")
    print(f"Query label: {labels[audio_id]}")
    print("\nRecommended similar sounds:\n")

    for match in results["matches"]:
        print("ID:", match["id"])
        print("Score:", match["score"])
        print("Label:", match["metadata"]["label"])
        print("---")

In [34]:
recommend_similar_audio(0, top_k=5)

Query audio ID: 0
Query label: dog

Recommended similar sounds:

ID: 0
Score: 1.00012076
Label: dog
---
ID: 77
Score: 0.876694322
Label: mouse_click
---
ID: 14
Score: 0.777210832
Label: dog
---
ID: 85
Score: 0.767594755
Label: coughing
---
ID: 84
Score: 0.754443586
Label: coughing
---


In [35]:
from IPython.display import Audio as IPythonAudio, display

def play_audio(audio_id):
    display(IPythonAudio(audios[audio_id], rate=32000))

In [36]:
play_audio(0)

In [37]:
play_audio(77)

In [38]:
def recommend_and_play(audio_id, top_k=5):
    query_vector = embeddings[audio_id]

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    print(f"Original audio ID: {audio_id}")
    print(f"Original label: {labels[audio_id]}")
    play_audio(audio_id)

    print("\nRecommended sounds:\n")

    for match in results["matches"]:
        match_id = int(match["id"])

        print("ID:", match_id)
        print("Score:", match["score"])
        print("Label:", match["metadata"]["label"])
        play_audio(match_id)
        print("---")

In [39]:
recommend_and_play(0, top_k=5)

Original audio ID: 0
Original label: dog



Recommended sounds:

ID: 0
Score: 1.00012076
Label: dog


---
ID: 77
Score: 0.876694322
Label: mouse_click


---
ID: 14
Score: 0.777210832
Label: dog


---
ID: 85
Score: 0.767594755
Label: coughing


---
ID: 84
Score: 0.754443586
Label: coughing


---


In [40]:
def recommend_without_self(audio_id, top_k=5):
    query_vector = embeddings[audio_id]

    results = index.query(
        vector=query_vector,
        top_k=top_k + 1,
        include_metadata=True
    )

    print(f"Original audio ID: {audio_id}")
    print(f"Original label: {labels[audio_id]}")
    play_audio(audio_id)

    print("\nRecommended sounds:\n")

    shown = 0

    for match in results["matches"]:
        match_id = int(match["id"])

        if match_id == audio_id:
            continue

        print("ID:", match_id)
        print("Score:", match["score"])
        print("Label:", match["metadata"]["label"])
        play_audio(match_id)
        print("---")

        shown += 1
        if shown == top_k:
            break

In [41]:
recommend_without_self(0, top_k=5)

Original audio ID: 0
Original label: dog



Recommended sounds:

ID: 77
Score: 0.876694322
Label: mouse_click


---
ID: 14
Score: 0.777210832
Label: dog


---
ID: 85
Score: 0.767594755
Label: coughing


---
ID: 84
Score: 0.754443586
Label: coughing


---
ID: 68
Score: 0.749836385
Label: laughing


---


## Conclusion

In this lab, we built an audio recommendation system using vector search.

First, we loaded audio samples from the ESC-50 dataset. Because the audio files were stored as bytes instead of local paths, we decoded them manually using `soundfile`.

Then, we used the PANNs audio tagging model to convert every audio clip into a numerical embedding. These embeddings capture the semantic characteristics of the sounds.

After that, we stored the embeddings in Pinecone, using cosine similarity to search for similar audio clips.

Finally, we queried Pinecone with a selected audio clip and retrieved the most similar sounds from the dataset. This shows how vector databases can be used to build recommendation systems for audio, music, or sound search applications.